# Historical Simulation & GBM-Predicted Risk Metrics
**Author:** Dev  
**Branch:** `feature/risk-metrics`  
**Project:** Empirical Portfolio Risk Modeling via Monte Carlo Simulation  
**Last updated:** Week 3

---

## Purpose

This notebook implements all of Dev's Week 1–3 deliverables for the `feature/risk-metrics` branch:

| # | Deliverable | Section |
|---|-------------|--------|
| 1 | Historical simulation VaR at 95% and 99% | §3 |
| 2 | Historical simulation ES at 95% and 99% | §4 |
| 3 | GBM-predicted VaR and ES (direct comparison) | §5 |
| 4 | Summary comparison table (all metrics, all 5 equities) | §6 |
| 5 | Return distribution histograms with VaR threshold lines | §7 |
| 6 | QQ-plots for all 5 equities | §8 |
| 7 | Excess kurtosis table (fat-tail diagnostic) | §9 |
| 8 | Tail Coverage Ratio (TCR) — novelty metric | §10 |

**Self-contained:** reads Nihan's CSVs from `/data/`. All figures saved to `/results/` before `plt.show()` per project guidelines §6a. Run `Kernel → Restart & Run All` to reproduce everything from scratch.

## 0. Imports and Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

NOTEBOOK_DIR = Path().resolve()
REPO_ROOT    = NOTEBOOK_DIR.parents[1]
DATA_DIR     = REPO_ROOT / 'data'
RESULTS_DIR  = REPO_ROOT / 'results'
QQ_DIR       = RESULTS_DIR / 'qq_plots'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
QQ_DIR.mkdir(parents=True, exist_ok=True)

TICKERS           = ['AAPL', 'AMZN', 'GOOGL', 'MSFT', 'TSLA']
CONFIDENCE_LEVELS = [0.95, 0.99]
N_PATHS           = 10_000
TRADING_DAYS      = 252

print(f'Repo root : {REPO_ROOT}')
print(f'Data dir  : {DATA_DIR}')
print(f'Results   : {RESULTS_DIR}')

## 1. Data Loading

Reads Nihan's five cleaned CSVs from `/data/`. Each file contains 5 years of daily OHLCV data. We extract closing prices and compute log-returns for all five equities.

In [ ]:
prices      = {}
log_returns = {}

for t in TICKERS:
    csv_path = DATA_DIR / f'{t}_daily_5y.csv'
    if not csv_path.exists():
        raise FileNotFoundError(
            f'{csv_path} not found. '
            'Rebase feature/risk-metrics onto dev so Nihan\'s CSVs are available.'
        )
    df = pd.read_csv(csv_path, index_col='Date', parse_dates=True)
    close_col = 'Adj Close' if 'Adj Close' in df.columns else 'Close'
    p = df[close_col].dropna()
    prices[t]      = p
    log_returns[t] = np.log(p / p.shift(1)).dropna()
    print(f'{t}: {len(p):,} obs | {p.index[0].date()} to {p.index[-1].date()}')

print('\n All five equities loaded from Nihan CVSs.')

## 2. Log-Return Definition

### Formula

The **daily log-return** at time $t$:

$$
r_t = \ln\!\left(\frac{S_t}{S_{t-1}}\right)
$$

Log-returns are time-additive, approximately symmetric, and analytically tractable under GBM. They are the standard input to both historical simulation and the GBM calibration pipeline. Already computed in §1 above.

## 3. Historical Simulation — Value at Risk

### Definition

**Value at Risk at confidence level $\alpha$** is the minimum loss on the worst $(1-\alpha)$% of trading days:

$$
\text{VaR}_\alpha = -\inf\{x \in \mathbb{R} : P(R \leq x) \geq 1-\alpha\}
$$

For **historical simulation**, we read directly from the empirical distribution — no distributional assumptions:

$$
\widehat{\text{VaR}}_\alpha^{\text{hist}} = -Q_{1-\alpha}(r_1, r_2, \ldots, r_T)
$$

where $Q_{1-\alpha}$ is the $(1-\alpha)$-th empirical quantile.

In [ ]:
def hist_var(returns, confidence):
    """Historical simulation VaR — positive loss magnitude."""
    return float(-np.quantile(returns, 1 - confidence))

print('AAPL Historical VaR:')
for cl in CONFIDENCE_LEVELS:
    v = hist_var(log_returns['AAPL'], cl)
    print(f'  VaR ({cl:.0%}) = {v:.4f}  ({v:.2%})')

## 4. Historical Simulation — Expected Shortfall

### Definition

**Expected Shortfall (ES / CVaR)** is the average loss given that we are in the tail beyond VaR. It is a **coherent risk measure** (Artzner et al., 1999), unlike VaR.

Integral form:

$$
\text{ES}_\alpha = -\frac{1}{1-\alpha}\int_0^{1-\alpha} Q_u(R)\, du
$$

Discrete empirical form:

$$
\widehat{\text{ES}}_\alpha^{\text{hist}} = -\mathbb{E}\bigl[R \mid R < -\widehat{\text{VaR}}_\alpha\bigr]
= -\frac{1}{|\mathcal{T}|}\sum_{r_t \in \mathcal{T}} r_t
$$

where $\mathcal{T} = \{r_t : r_t < -\widehat{\text{VaR}}_\alpha\}$.

In [ ]:
def hist_es(returns, confidence):
    """Historical simulation ES — positive loss magnitude."""
    threshold = -hist_var(returns, confidence)
    tail = returns[returns < threshold]
    return float(-tail.mean()) if len(tail) > 0 else np.nan

print('AAPL Historical ES:')
for cl in CONFIDENCE_LEVELS:
    e = hist_es(log_returns['AAPL'], cl)
    n_tail = (log_returns['AAPL'] < -hist_var(log_returns['AAPL'], cl)).sum()
    print(f'  ES ({cl:.0%}) = {e:.4f}  ({e:.2%})  [tail n = {n_tail}]')

## 5. GBM-Simulated VaR and ES

### Motivation

The GBM model assumes log-returns are normally distributed:

$$
r_t \sim \mathcal{N}\!\left(\left(\mu - \frac{\sigma^2}{2}\right)\Delta t,\ \sigma^2 \Delta t\right)
$$

Under this assumption, the GBM discrete update equation is:

$$
S(t + \Delta t) = S(t)\exp\!\left[\left(\mu - \frac{\sigma^2}{2}\right)\Delta t + \sigma\sqrt{\Delta t}\, Z\right], \quad Z \sim \mathcal{N}(0,1)
$$

### Calibration

We calibrate $\hat{\mu}$ and $\hat{\sigma}$ from the first four years of each equity's data, matching Robert's approach in `src/simulations/sim.py`:

$$
\hat{\mu} = \bar{r} \times 252, \qquad \hat{\sigma} = s_r \times \sqrt{252}
$$

We then simulate 10,000 paths and extract the implied distribution of **daily log-returns** for comparison against historical simulation at the same daily horizon.

**Note:** The `GBM` class below mirrors Robert's `src/models/gbm.py`. After rebasing onto `dev`, replace the class definition with `from src.models.gbm import GBM`.

In [ ]:
# from src.models.gbm import GBM  # uncomment after rebasing onto dev

class GBM:
    """Mirrors src/models/gbm.py. Replace with import after rebase."""
    def __init__(self, S0, mu, sigma, T=1.0, N=252, seed=42):
        self.S0, self.mu, self.sigma = S0, mu, sigma
        self.T, self.N, self.seed = T, N, seed
        self.dt = T / N

    def simulate(self, paths=10_000):
        """Returns price paths array of shape (paths, N+1)."""
        rng = np.random.default_rng(self.seed)
        Z   = rng.normal(size=(paths, self.N))
        inc = (self.mu - 0.5 * self.sigma**2) * self.dt + self.sigma * np.sqrt(self.dt) * Z
        return np.hstack([self.S0 * np.ones((paths, 1)), self.S0 * np.exp(np.cumsum(inc, axis=1))])

    def simulate_daily_log_returns(self, paths=10_000):
        """
        Returns flattened array of simulated daily log-returns (paths x N obs).
        Used for daily VaR/ES comparison against historical simulation.
        """
        rng = np.random.default_rng(self.seed)
        Z   = rng.normal(size=(paths, self.N))
        return ((self.mu - 0.5*self.sigma**2)*self.dt + self.sigma*np.sqrt(self.dt)*Z).flatten()


def calibrate_gbm(ticker):
    """Calibrate mu and sigma from first 4 years of data."""
    p     = prices[ticker]
    total = len(p)
    lr_4y = np.log(p.iloc[total - TRADING_DAYS*4 : total - TRADING_DAYS] /
                   p.iloc[total - TRADING_DAYS*4 : total - TRADING_DAYS].shift(1)).dropna()
    return float(lr_4y.mean() * TRADING_DAYS), float(lr_4y.std() * np.sqrt(TRADING_DAYS))


def gbm_var_es(ticker, confidence):
    """Simulate GBM daily log-returns and compute VaR + ES."""
    mu, sigma = calibrate_gbm(ticker)
    gbm       = GBM(S0=float(prices[ticker].iloc[0]), mu=mu, sigma=sigma,
                    T=1.0, N=TRADING_DAYS, seed=42)
    sim_lr    = gbm.simulate_daily_log_returns(paths=N_PATHS)
    var_val   = float(-np.quantile(sim_lr, 1 - confidence))
    tail      = sim_lr[sim_lr < -var_val]
    es_val    = float(-tail.mean()) if len(tail) > 0 else np.nan
    return var_val, es_val, mu, sigma


print('AAPL GBM-predicted VaR and ES:')
for cl in CONFIDENCE_LEVELS:
    v, e, mu, sig = gbm_var_es('AAPL', cl)
    print(f'  VaR ({cl:.0%}) = {v:.4f} ({v:.2%}) | ES ({cl:.0%}) = {e:.4f} ({e:.2%})')
print(f'  Calibrated: mu = {mu:.4f}, sigma = {sig:.4f}')

## 6. Full Risk Metrics Summary Table

Historical vs GBM-predicted VaR and ES at both confidence levels for all five equities. The **GBM/Hist VaR** column quantifies GBM's underestimation of tail risk. Values below 1.0 mean GBM predicts a smaller loss threshold than history actually delivers — the expected result given fat tails.

This table goes directly into Section 6 (Risk Metric Results) of the paper.

In [ ]:
rows = []
gbm_cache = {}

for t in TICKERS:
    r = log_returns[t]
    for cl in CONFIDENCE_LEVELS:
        h_var = hist_var(r, cl)
        h_es  = hist_es(r, cl)
        g_var, g_es, mu_c, sig_c = gbm_var_es(t, cl)
        gbm_cache[(t, cl)] = (g_var, g_es, mu_c, sig_c)
        n_breach = int((r < -h_var).sum())
        tcr = n_breach / ((1 - cl) * len(r))
        rows.append({
            'Ticker'        : t,
            'Conf.'         : f'{cl:.0%}',
            'Hist VaR %'    : f'{h_var:.2%}',
            'GBM VaR %'     : f'{g_var:.2%}',
            'GBM/Hist VaR'  : f'{g_var/h_var:.3f}',
            'Hist ES %'     : f'{h_es:.2%}',
            'GBM ES %'      : f'{g_es:.2%}',
            'TCR'           : f'{tcr:.3f}',
        })

results_df = pd.DataFrame(rows)
print('=' * 78)
print('  Historical vs GBM Risk Metrics — All Five Equities')
print('=' * 78)
print(results_df.to_string(index=False))
print('=' * 78)
results_df

## 7. Return Distribution Histograms

For each equity: histogram of empirical daily log-returns with vertical VaR threshold lines at 95% (red dashed) and 99% (dark red dotted), plus a fitted normal distribution overlay to visualise fat-tail deviation. Saved to `/results/` at 300 dpi.

In [ ]:
for t in TICKERS:
    r = log_returns[t]
    fig, ax = plt.subplots(figsize=(12, 6))
    fig.patch.set_facecolor('#f9f9f9')
    ax.set_facecolor('#f9f9f9')

    counts, bins, patches = ax.hist(
        r.values, bins=80, density=True,
        color='#1e3a5f', alpha=0.70, edgecolor='white', linewidth=0.3,
        label=f'{t} empirical log-returns'
    )

    v95, v99 = hist_var(r, 0.95), hist_var(r, 0.99)
    for patch, left in zip(patches, bins[:-1]):
        if left < -v99:
            patch.set_facecolor('#7b0000'); patch.set_alpha(0.80)
        elif left < -v95:
            patch.set_facecolor('#e74c3c'); patch.set_alpha(0.75)

    mu_h, sig_h = float(r.mean()), float(r.std())
    x_r = np.linspace(float(r.min()) * 1.3, float(r.max()) * 1.3, 500)
    ax.plot(x_r, stats.norm.pdf(x_r, mu_h, sig_h),
            color='#555', lw=1.5, ls='--', alpha=0.8,
            label=f'Normal fit (μ={mu_h:.4f}, σ={sig_h:.4f})')

    ymax = counts.max()
    for cl, col, ls, yf in [(0.95, '#e74c3c', '--', 0.58), (0.99, '#7b0000', ':', 0.40)]:
        v = hist_var(r, cl)
        ax.axvline(-v, color=col, lw=2.0, ls=ls, label=f'VaR {cl:.0%} = {v:.2%}')
        ax.annotate(f'VaR {cl:.0%}\n{v:.2%}',
                    xy=(-v, ymax*0.25), xytext=(-v-0.008, ymax*yf),
                    fontsize=9, color=col, ha='right',
                    arrowprops=dict(arrowstyle='->', color=col, lw=1.2))

    start = prices[t].index[0].strftime('%Y-%m-%d')
    end   = prices[t].index[-1].strftime('%Y-%m-%d')
    ax.set_title(f'Empirical Log-Return Distribution — {t}\nHistorical Simulation VaR | {start} to {end}',
                 fontsize=13, fontweight='bold', pad=12)
    ax.set_xlabel(r'Daily Log-Return $r_t = \ln(S_t/S_{t-1})$', fontsize=11)
    ax.set_ylabel('Probability Density', fontsize=11)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=1))
    ax.legend(fontsize=9, framealpha=0.85, loc='upper left')
    ax.grid(axis='y', alpha=0.3); ax.grid(axis='x', alpha=0.2)
    ax.text(0.98, 0.97,
            f'n={len(r):,}\nSkew={float(r.skew()):.3f}\nKurt={float(r.kurt()):.3f}',
            transform=ax.transAxes, fontsize=9, va='top', ha='right',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.75))
    plt.tight_layout()
    fig_path = RESULTS_DIR / f'{t}_log_return_distribution_VaR_ES.png'
    fig.savefig(fig_path, dpi=300, bbox_inches='tight')  # save BEFORE show()
    plt.show()
    print(f'  Saved -> {fig_path.name}')

## 8. QQ-Plots — Normality Diagnostic

### What Is a QQ-Plot?

A **quantile-quantile (QQ) plot** compares the empirical quantiles of a sample against theoretical normal quantiles. Points on the red diagonal line = perfect normality.

**Interpretation for equity log-returns:**
- **S-curve shape** (points curving away from the line at both ends) = **fat tails** (positive excess kurtosis)
- **Asymmetric deviation** = skewness

Both patterns motivate using historical simulation rather than parametric GBM for tail risk estimation, and feed directly into Section 7 (Limitations) of the paper.

Individual figures saved to `/results/qq_plots/`. Grid overview saved to `/results/qq_all_equities.png`.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('QQ-Plots: Empirical Log-Returns vs Normal Distribution — All Five Equities',
             fontsize=14, fontweight='bold', y=1.01)
axes_flat = axes.flatten()

for i, t in enumerate(TICKERS):
    ax = axes_flat[i]
    ax.set_facecolor('#f9f9f9')
    r = log_returns[t].values

    (osm, osr), (slope, intercept, r_val) = stats.probplot(r, dist='norm')
    ax.scatter(osm, osr, color='#1e3a5f', s=5, alpha=0.45, label='Empirical quantiles')
    x_line = np.array([min(osm), max(osm)])
    ax.plot(x_line, slope*x_line + intercept, 'r--', lw=1.8, label='Normal reference')

    kv = float(log_returns[t].kurt())
    sv = float(log_returns[t].skew())
    ax.set_title(f'{t}  (R\u00b2 = {r_val**2:.4f})', fontsize=11, fontweight='bold')
    ax.set_xlabel('Theoretical Normal Quantiles', fontsize=9)
    ax.set_ylabel('Sample Quantiles', fontsize=9)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.text(0.04, 0.96, f'Excess Kurt = {kv:.3f}\nSkewness = {sv:.3f}',
            transform=ax.transAxes, fontsize=8.5, va='top',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

    # Save individual QQ plot
    fig_ind, ax_ind = plt.subplots(figsize=(6, 6))
    ax_ind.set_facecolor('#f9f9f9')
    ax_ind.scatter(osm, osr, color='#1e3a5f', s=6, alpha=0.5, label='Empirical quantiles')
    ax_ind.plot(x_line, slope*x_line + intercept, 'r--', lw=1.8, label='Normal reference')
    ax_ind.set_title(f'QQ-Plot: {t} Daily Log-Returns vs Normal\n(R\u00b2 = {r_val**2:.4f})',
                     fontsize=12, fontweight='bold')
    ax_ind.set_xlabel('Theoretical Normal Quantiles', fontsize=10)
    ax_ind.set_ylabel('Sample Quantiles', fontsize=10)
    ax_ind.legend(fontsize=9); ax_ind.grid(alpha=0.3)
    ax_ind.text(0.04, 0.96, f'Excess Kurt = {kv:.3f}\nSkewness = {sv:.3f}',
                transform=ax_ind.transAxes, fontsize=9, va='top',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))
    plt.tight_layout()
    ind_path = QQ_DIR / f'qq_{t}.png'
    fig_ind.savefig(ind_path, dpi=300, bbox_inches='tight')
    plt.close(fig_ind)
    print(f'  Individual QQ saved -> qq_plots/qq_{t}.png')

axes_flat[5].set_visible(False)
plt.tight_layout()
grid_path = RESULTS_DIR / 'qq_all_equities.png'
fig.savefig(grid_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Grid saved -> {grid_path.name}')

## 9. Excess Kurtosis Table

### Definition

**Excess kurtosis** measures how much heavier the tails are relative to the normal distribution:

$$
\kappa_{\text{excess}} = \frac{1}{\sigma^4}\int_{-\infty}^{\infty}(r - \mu)^4 f(r)\, dr - 3
$$

For a normal distribution, $\kappa_{\text{excess}} = 0$. **Positive excess kurtosis** (leptokurtosis) means fatter tails than normal — more extreme returns than GBM predicts. This is the root cause of GBM's systematic VaR underestimation at the 99% level.

This table is a **key result** for Section 7 (Limitations and Failure Modes) of the paper.

In [ ]:
kurt_rows = []
for t in TICKERS:
    r = log_returns[t]
    kurt_rows.append({
        'Ticker'          : t,
        'Obs (n)'         : len(r),
        'Mean (daily)'    : round(float(r.mean()), 5),
        'Std (daily)'     : round(float(r.std()),  5),
        'Ann. Vol'        : f'{float(r.std() * np.sqrt(252)):.2%}',
        'Skewness'        : round(float(r.skew()), 4),
        'Excess Kurtosis' : round(float(r.kurt()), 4),
        'Fat Tails?'      : 'Yes' if r.kurt() > 0 else 'No',
    })

kurt_df = pd.DataFrame(kurt_rows).set_index('Ticker')
print('=' * 72)
print('  Excess Kurtosis & Distributional Properties')
print('=' * 72)
print(kurt_df.to_string())
print('=' * 72)
print('All equities show positive excess kurtosis -> fat tails -> GBM underestimates VaR at 99%')

kurt_df

## 10. Tail Coverage Ratio (TCR) — Novel Metric

### Definition

The **Tail Coverage Ratio** is the paper's key novelty contribution. It quantifies GBM's failure as a single interpretable scalar per equity and confidence level:

$$
\text{TCR}_\alpha = \frac{\text{Realized breach frequency}}{\text{Theoretical breach frequency under GBM}}
$$

- **Realized breach frequency** = fraction of historical days where $r_t < -\widehat{\text{VaR}}_\alpha^{\text{GBM}}$
- **Theoretical breach frequency** = $1 - \alpha$ (e.g. 5% at 95%)

Under a correctly specified GBM, $\text{TCR} = 1.0$. A value $> 1.0$ means the realized tail is fatter than GBM predicts. **The hypothesis is that TCR is systematically $> 1.0$ across all equities and both confidence levels, and that TCR correlates positively with excess kurtosis.**

In [ ]:
tcr_rows = []
for t in TICKERS:
    r = log_returns[t]
    for cl in CONFIDENCE_LEVELS:
        g_var, _, _, _ = gbm_cache[(t, cl)]
        n_total       = len(r)
        n_breach      = int((r < -g_var).sum())
        realized_freq = n_breach / n_total
        theo_freq     = 1 - cl
        tcr           = realized_freq / theo_freq
        tcr_rows.append({
            'Ticker'           : t,
            'Conf.'            : f'{cl:.0%}',
            'GBM VaR'          : f'{g_var:.2%}',
            'Hist VaR'         : f'{hist_var(r, cl):.2%}',
            'Realized breaches': n_breach,
            'Realized freq'    : f'{realized_freq:.3%}',
            'Theo. freq'       : f'{theo_freq:.1%}',
            'TCR'              : round(tcr, 4),
            'Excess Kurt'      : round(float(r.kurt()), 3),
            'GBM fails?'       : 'YES' if tcr > 1 else 'No',
        })

tcr_df = pd.DataFrame(tcr_rows)
print('=' * 90)
print('  Tail Coverage Ratio — TCR > 1.0 means realized breaches exceed GBM prediction')
print('=' * 90)
print(tcr_df.to_string(index=False))
print('=' * 90)
tcr_df

## 11. Reproducibility Checklist

Before opening the PR into `dev`:

- [ ] Rebased `feature/risk-metrics` onto `dev` — all Nihan CSV paths resolve without fallback
- [ ] `Kernel → Restart & Run All` — zero errors
- [ ] `Kernel → Restart & Clear Output` before committing
- [ ] `/results/` has histograms for all 5 equities at 300 dpi
- [ ] `/results/qq_plots/` has individual QQ plots for all 5 equities at 300 dpi
- [ ] `/results/qq_all_equities.png` grid figure present
- [ ] Commit message: `[risk] extend notebook with GBM comparison, QQ plots, kurtosis, TCR`
- [ ] PR opened into `dev`, review requested from Ayush

---

## References

- Artzner, P., Delbaen, F., Eber, J.-M., & Heath, D. (1999). Coherent measures of risk. *Mathematical Finance*, 9(3), 203–228.
- Basel Committee on Banking Supervision (2019). *Minimum Capital Requirements for Market Risk* (FRTB). BIS.
- Hull, J. C. (2018). *Risk Management and Financial Institutions* (5th ed.). Wiley.
- McNeil, A. J., Frey, R., & Embrechts, P. (2015). *Quantitative Risk Management* (Rev. ed.). Princeton UP.